## Setup

In [1]:
from agent_base.media_backend.local import LocalMediaBackend
from pprint import pprint

from agent_base.sandbox.local import LocalSandbox

MEDIA_BASE_PREFIX = "/media"
media_backend = LocalMediaBackend("./agent-media", f"{MEDIA_BASE_PREFIX}/")

sandbox = LocalSandbox('id-123', './sandbox-123')
await sandbox.setup()

media_backend.attach_sandbox(sandbox)
await media_backend.connect()

In [2]:
from typing import AsyncIterator

async def hello_generator(n: int = 5) -> AsyncIterator[bytes]:
    for i in range(n):
        yield f"Hello World! (line {i+1})\n".encode("utf-8")


async def consume(stream: AsyncIterator[bytes]) -> str:
    chunks: list[bytes] = []
    async for chunk in stream:
        chunks.append(chunk)
    return b"".join(chunks).decode("utf-8")

## Storage Operations

In [12]:
# Store
store_res = await media_backend.store(
    hello_generator(),
    "hello.txt",
    "text/plain",
    "id-123"
)

pprint(store_res)

MediaMetadata(media_id='30edb3d85a1c40f4bbc0f922228b8ee8',
              media_mime_type='text/plain',
              media_filename='hello.txt',
              media_extension='txt',
              media_size=110,
              storage_type='local',
              storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/30edb3d85a1c40f4bbc0f922228b8ee8_hello.txt',
              extras={})


In [13]:
# Retrieve
retrieve_res = await consume(media_backend.retrieve(
    store_res.media_id,
    "id-123"
))

pprint(retrieve_res)

('Hello World! (line 1)\n'
 'Hello World! (line 2)\n'
 'Hello World! (line 3)\n'
 'Hello World! (line 4)\n'
 'Hello World! (line 5)\n')


In [14]:
# Exists
exists_res = await media_backend.exists(
    store_res.media_id,
    "id-123"
)

pprint(exists_res)

(True,
 MediaMetadata(media_id='30edb3d85a1c40f4bbc0f922228b8ee8',
               media_mime_type='text/plain',
               media_filename='hello.txt',
               media_extension='txt',
               media_size=110,
               storage_type='local',
               storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/30edb3d85a1c40f4bbc0f922228b8ee8_hello.txt',
               extras={}))


In [15]:
update_metadata_res = await media_backend.update_metadata(
    store_res.media_id,
    "id-123",
    {"key": "value"}
)

pprint(update_metadata_res)

None


In [16]:
# Get Metadata
metadata_res = await media_backend.get_metadata(
    store_res.media_id,
    "id-123"
)

pprint(metadata_res)

MediaMetadata(media_id='30edb3d85a1c40f4bbc0f922228b8ee8',
              media_mime_type='text/plain',
              media_filename='hello.txt',
              media_extension='txt',
              media_size=110,
              storage_type='local',
              storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/30edb3d85a1c40f4bbc0f922228b8ee8_hello.txt',
              extras={'key': 'value'})


In [17]:
# Delete
delete_res = await media_backend.delete(
    store_res.media_id,
    "id-123"
)

pprint(delete_res)

True


## Sandbox Bridge

In [18]:
# User Upload
user_upload_res = await media_backend.user_upload(
    hello_generator(10),
    "hello.txt",
    "text/plain",
    "id-123"
)

In [21]:
store_res_2 = await media_backend.store(
    hello_generator(5),
    "hello2.txt",
    "text/plain",
    "id-123"
)

pprint(store_res_2)


MediaMetadata(media_id='5f4d52ab16d44fa8823a8edc06b58597',
              media_mime_type='text/plain',
              media_filename='hello2.txt',
              media_extension='txt',
              media_size=110,
              storage_type='local',
              storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/5f4d52ab16d44fa8823a8edc06b58597_hello2.txt',
              extras={})


In [22]:
materialized_res = await media_backend.materialize(
    store_res_2.media_id,
    "id-123"
)

pprint(materialized_res)


'.imported/hello2.txt'


In [27]:
sandbox_export = await media_backend._sandbox.write_file_bytes(
    "./.exports/hello3.txt",
    hello_generator(3)
)

pprint(sandbox_export)

None


In [28]:
flush_res = await media_backend.flush_exports(
    "id-123"
)

pprint(flush_res)

[MediaMetadata(media_id='dae4b0b13d584307b6ae1b541235bef7',
               media_mime_type='text/plain',
               media_filename='hello3.txt',
               media_extension='txt',
               media_size=66,
               storage_type='local',
               storage_location='/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/dae4b0b13d584307b6ae1b541235bef7_hello3.txt',
               extras={'blake3_hash': '2afee12deedfa4b43391e67e13e3be27f9a972620e444b2e873e17b0b6af4d8c'})]


## Transport Layer Conversion

In [30]:
url_res = await media_backend.to_url(
    flush_res[0].media_id,
    "id-123"
)

pprint(url_res)

'/media/id-123/dae4b0b13d584307b6ae1b541235bef7'


In [31]:
reference_res = await media_backend.to_reference(
    flush_res[0].media_id,
    "id-123"
)

pprint(reference_res)

{'extras': {},
 'media_extension': 'txt',
 'media_filename': 'hello3.txt',
 'media_id': 'dae4b0b13d584307b6ae1b541235bef7',
 'media_mime_type': 'text/plain',
 'media_size': 66,
 'storage_location': '/Users/aurosoni/conductor/workspaces/anthropic_agent/quebec/agent_base/media_backend/agent-media/id-123/dae4b0b13d584307b6ae1b541235bef7_hello3.txt',
 'storage_type': 'local'}


In [32]:
bytes_res = await media_backend.to_base64(
    flush_res[0].media_id,
    "id-123"
)

pprint(bytes_res)            

{'content': b'Hello World! (line 1)\nHello World! (line 2)\nHello World! (li'
            b'ne 3)\n',
 'mime_type': 'text/plain'}
